In [1]:
import warnings
warnings.filterwarnings("ignore")

## Step 1: Load Pre-split CSVs and Clean Text

In [2]:
import pandas as pd
import re
import string

# ================================
# Dataset Paths
# ================================
TRAIN_CSV = "/kaggle/input/datasets/ariful234/banclickthumb-bangla-clickbait-dataset/BanClickThumb-Bangla Clickbait Dataset/train.csv"
VAL_CSV   = "/kaggle/input/datasets/ariful234/banclickthumb-bangla-clickbait-dataset/BanClickThumb-Bangla Clickbait Dataset/val.csv"
TEST_CSV  = "/kaggle/input/datasets/ariful234/banclickthumb-bangla-clickbait-dataset/BanClickThumb-Bangla Clickbait Dataset/test.csv"

# ================================
# Text Cleaning Functions
# ================================
def remove_punctuation(text):
    if pd.isna(text):
        return text
    return text.translate(str.maketrans('', '', string.punctuation))

def remove_whitespace(text):
    if pd.isna(text):
        return text
    return " ".join(text.split())

def remove_emojis(text):
    if pd.isna(text):
        return text
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def remove_urls(text):
    if pd.isna(text):
        return text
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def remove_html(text):
    if pd.isna(text):
        return text
    html_pattern = re.compile(r'<.*?>')
    return html_pattern.sub(r'', text)

def remove_special_characters(text):
    if pd.isna(text):
        return text
    return re.sub(r'[^A-Za-z0-9\s\u0980-\u09FF]', '', text)

def clean_text(text):
    text = remove_urls(text)
    text = remove_html(text)
    text = remove_emojis(text)
    text = remove_punctuation(text)
    text = remove_special_characters(text)
    text = remove_whitespace(text)
    return text

# ================================
# Load & Clean Splits
# ================================
splits = {
    'Train':      TRAIN_CSV,
    'Validation': VAL_CSV,
    'Test':       TEST_CSV,
}

cleaned_paths = {
    'Train':      '/kaggle/working/train_cleaned.csv',
    'Validation': '/kaggle/working/val_cleaned.csv',
    'Test':       '/kaggle/working/test_cleaned.csv',
}

for key, csv_path in splits.items():
    df = pd.read_csv(csv_path)
    df = df[['Title', 'label']]
    df = df.dropna(subset=['label']).reset_index(drop=True)
    df['Title'] = df['Title'].astype(str).apply(clean_text)
    df['label'] = df['label'].astype(int)
    df.to_csv(cleaned_paths[key], index=False)
    print(f"{key} cleaned & saved → {cleaned_paths[key]}")
    print(df['label'].value_counts(), "\n")

Train cleaned & saved → /kaggle/working/train_cleaned.csv
label
0    2886
1    2116
Name: count, dtype: int64 

Validation cleaned & saved → /kaggle/working/val_cleaned.csv
label
0    412
1    303
Name: count, dtype: int64 

Test cleaned & saved → /kaggle/working/test_cleaned.csv
label
0    825
1    605
Name: count, dtype: int64 



In [3]:
train_df = pd.read_csv('/kaggle/working/train_cleaned.csv')
train_df.head()

,Title,label
0,ব্রেন স্ট্রোকের লক্ষণ গুলি কি Brain Stroke Sym...,1
1,২য় শ্রেণি গণিত নমুনা প্রশ্ন ও উত্তর ২০২৫ ১ম স...,0
2,চ্যালেঞ্জ সামলাতে পারবে যুক্তরাষ্ট্রের সাথে সম...,1
3,কুমির চাষে আয় ১৫ কোটি টাকা না দেখলে বিশ্বাস ক...,1
4,আ লীগ নির্বাচনে থাকবে কিনা সিদ্ধান্ত কার ওপর ছ...,0


In [4]:
val_df = pd.read_csv('/kaggle/working/val_cleaned.csv')
val_df.head()

,Title,label
0,হাসিনার প্লট জালিয়াতি ফাঁসছেন ২০ কর্মকর্তা DU...,0
1,কক্সবাজারে এনসিপি নেতাদের গাড়িবহরে হামলা Coxs...,0
2,WBCHSE 2025 Results LIVE উচ্চমাধ্যমিকের ফলাফল ...,0
3,কাল থেকে পুরোদমে শুরু হচ্ছে দেশের শিক্ষা কার্য...,1
4,নবম শ্রেণীর পাঠ্যবই নিয়ে যে তথ্য দিলেন আসিফ ম...,0


In [5]:
test_df = pd.read_csv('/kaggle/working/test_cleaned.csv')
test_df.head()

,Title,label
0,অহংকারের পরিণতি কি মুফতি সাইফুল্লাহ মাহমুদআল্ল...,0
1,ইলেকট্রিক কেটলিতে পানি গরম করলে কত টাকা বিদ্যু...,1
2,বনসংরক্ষণ প্রবন্ধ রচনা মাধ্যমিক ও উচ্চমাধ্যমিক...,0
3,টপ ৫ ইতিহাসের সবচেয়ে ভয়াবহ প্রাকৃতিক দুর্যোগ...,1
4,পাকিস্তানিরা কেন শেখ মুজিবের চেয়ে তাজউদ্দীনকে...,1


## Step 2: Install & Import Dependencies

In [6]:
!pip install transformers -q

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import numpy as np

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cuda


## Step 3: Tokenizer & Dataset

In [8]:
# ================================
# DistilBERT Model Name
# ================================
MODEL_NAME = "distilbert-base-multilingual-cased"
MAX_LEN    = 128
BATCH_SIZE = 32

tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: distilbert-base-multilingual-cased


In [9]:
class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts     = df['Title'].tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        # DistilBERT does NOT use token_type_ids
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = TextDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = TextDataset(val_df,   tokenizer, MAX_LEN)
test_dataset  = TextDataset(test_df,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Train: 5002 | Val: 715 | Test: 1430


## Step 4: Define DistilBERT Classifier Model

In [10]:
class DistilBertClassifier(nn.Module):
    def __init__(self, model_name, num_labels=2, dropout=0.3):
        super(DistilBertClassifier, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.distilbert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        # DistilBERT returns last_hidden_state; use [CLS] token (index 0)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits     = self.classifier(cls_output)
        return logits

model = DistilBertClassifier(MODEL_NAME).to(DEVICE)
print(model)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertClassifier(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (l

## Step 5: Training Setup

In [11]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCHS        = 15
LEARNING_RATE = 2e-5
WARMUP_STEPS  = 0

optimizer   = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)
loss_fn = nn.CrossEntropyLoss().to(DEVICE)
print("Optimizer, scheduler, and loss function ready.")

Optimizer, scheduler, and loss function ready.


## Step 6: Train & Validate

In [12]:
def train_epoch(model, loader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = loss_fn(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            logits = model(input_ids, attention_mask)
            loss   = loss_fn(logits, labels)
            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = correct / total
    return avg_loss, acc, all_preds, all_labels, all_probs

In [13]:
best_val_loss = float('inf')
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, DEVICE)
    vl_loss, vl_acc, _, _, _ = eval_epoch(model, val_loader, loss_fn, DEVICE)

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    train_accs.append(tr_acc)
    val_accs.append(vl_acc)

    print(f"Epoch {epoch}/{EPOCHS} | "
          f"Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f} | Val Acc: {vl_acc:.4f}")

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save(model.state_dict(), '/kaggle/working/best_distilbert_model.pt')
        print("  ✓ Best model saved.")

Epoch 1/15 | Train Loss: 0.6127 | Train Acc: 0.6543 | Val Loss: 0.5897 | Val Acc: 0.6713
  ✓ Best model saved.
Epoch 2/15 | Train Loss: 0.4764 | Train Acc: 0.7747 | Val Loss: 0.5121 | Val Acc: 0.7650
  ✓ Best model saved.
Epoch 3/15 | Train Loss: 0.3450 | Train Acc: 0.8497 | Val Loss: 0.5796 | Val Acc: 0.7706
Epoch 4/15 | Train Loss: 0.2467 | Train Acc: 0.9048 | Val Loss: 0.6082 | Val Acc: 0.7678
Epoch 5/15 | Train Loss: 0.1606 | Train Acc: 0.9406 | Val Loss: 0.7763 | Val Acc: 0.7678
Epoch 6/15 | Train Loss: 0.1140 | Train Acc: 0.9622 | Val Loss: 1.0661 | Val Acc: 0.7622
Epoch 7/15 | Train Loss: 0.0891 | Train Acc: 0.9692 | Val Loss: 1.1785 | Val Acc: 0.7594
Epoch 8/15 | Train Loss: 0.0705 | Train Acc: 0.9808 | Val Loss: 1.2065 | Val Acc: 0.7762
Epoch 9/15 | Train Loss: 0.0619 | Train Acc: 0.9816 | Val Loss: 1.2683 | Val Acc: 0.7734
Epoch 10/15 | Train Loss: 0.0417 | Train Acc: 0.9884 | Val Loss: 1.4336 | Val Acc: 0.7720
Epoch 11/15 | Train Loss: 0.0373 | Train Acc: 0.9888 | Val Loss: 

## Step 7: Training Curves

In [14]:
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, train_losses, marker='o', label='Train Loss')
axes[0].plot(epochs_range, val_losses,   marker='s', label='Val Loss')
axes[0].set_title('Loss per Epoch (DistilBERT)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_range, train_accs, marker='o', label='Train Acc')
axes[1].plot(epochs_range, val_accs,   marker='s', label='Val Acc')
axes[1].set_title('Accuracy per Epoch (DistilBERT)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()
print("Training curves saved to /kaggle/working/training_curves.png")

Training curves saved to /kaggle/working/training_curves.png


## Step 8: Evaluate on Test Set

In [15]:
# Load best checkpoint
model.load_state_dict(torch.load('/kaggle/working/best_distilbert_model.pt', map_location=DEVICE))
model.eval()
print("Best model loaded for test evaluation.")

Best model loaded for test evaluation.


In [16]:
_, test_acc, test_preds, test_labels, test_probs = eval_epoch(model, test_loader, loss_fn, DEVICE)

precision = precision_score(test_labels, test_preds)
recall    = recall_score(test_labels, test_preds)
f1        = f1_score(test_labels, test_preds)
roc_auc   = roc_auc_score(test_labels, test_probs)

print("\n========== Test Set Results (DistilBERT Unimodal Text) ==========")
print(f"  Accuracy  : {test_acc:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  ROC-AUC   : {roc_auc:.4f}")
print("================================================================")


========== Test Set Results (DistilBERT Unimodal Text) ==========
  Accuracy  : 0.7818
  Precision : 0.7548
  Recall    : 0.7174
  F1-Score  : 0.7356
  ROC-AUC   : 0.8519


## Step 9: Confusion Matrix

In [17]:
import itertools

cm = confusion_matrix(test_labels, test_preds)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
classes = ['Non-Clickbait (0)', 'Clickbait (1)']
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=classes, yticklabels=classes,
       title='Confusion Matrix – DistilBERT Unimodal Text',
       ylabel='True Label',
       xlabel='Predicted Label')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')

thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, format(cm[i, j], 'd'),
            ha='center', va='center',
            color='white' if cm[i, j] > thresh else 'black')

fig.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()
print("Confusion matrix saved to /kaggle/working/confusion_matrix.png")

Confusion matrix saved to /kaggle/working/confusion_matrix.png


## Step 10: ROC Curve

In [18]:
fpr, tpr, _ = roc_curve(test_labels, test_probs)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve – DistilBERT Unimodal Text')
plt.legend(loc='lower right')
plt.grid(True)
plt.savefig('/kaggle/working/roc_curve.png', dpi=150)
plt.show()
print("ROC curve saved to /kaggle/working/roc_curve.png")

ROC curve saved to /kaggle/working/roc_curve.png
